# 06c — Langfuse LLMOps
**Domain:** Multi-domain &nbsp;|&nbsp; **Time:** ~2 h  
**Key Concepts:** distributed tracing, prompt versioning, A/B testing, cost monitoring

---
### Setup Langfuse (self-hosted, free)
```bash
git clone https://github.com/langfuse/langfuse.git && cd langfuse
docker compose up -d          # opens at http://localhost:3000
```
Create a project → copy **Public Key** + **Secret Key** into the cell below.


In [ ]:
import sys, os
sys.path.insert(0, '.')
from llm_checker import check, hint, evaluate, progress, show_exercise
print("✅ ready")


In [ ]:
try:
    from langfuse import Langfuse
    LF_INSTALLED = True
    print("✅ Langfuse installed")
except ImportError:
    LF_INSTALLED = False
    print("❌ pip install langfuse")

LANGFUSE_HOST       = os.getenv("LANGFUSE_HOST",       "http://localhost:3000")
LANGFUSE_PUBLIC_KEY = os.getenv("LANGFUSE_PUBLIC_KEY", "pk-lf-placeholder")
LANGFUSE_SECRET_KEY = os.getenv("LANGFUSE_SECRET_KEY", "sk-lf-placeholder")

if LF_INSTALLED:
    try:
        lf = Langfuse(public_key=LANGFUSE_PUBLIC_KEY,
                      secret_key=LANGFUSE_SECRET_KEY,
                      host=LANGFUSE_HOST)
        LF_AVAILABLE = True
        print(f"✅ Connected to Langfuse at {LANGFUSE_HOST}")
    except Exception as e:
        LF_AVAILABLE = False
        print(f"⚠️  Langfuse not reachable ({e}) — running in mock mode")
else:
    lf = None
    LF_AVAILABLE = False


---
## 🔵 Example — Ex 06c-1: First Langfuse Trace with 3 Spans

In [ ]:
from openai import OpenAI
import time

client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

def traced_rag_call(question: str, trace_name: str = "rag-query") -> str:
    """RAG call with 3 Langfuse spans: retrieve → rerank → generate."""
    trace = lf.trace(name=trace_name, metadata={"question": question}) if LF_AVAILABLE else None
    t0 = time.perf_counter()

    # Span 1 — Retrieve
    context = "Transformers use self-attention to process sequences in parallel."
    if trace:
        s1 = trace.span(name="retrieve", input={"query": question}, output={"context": context})
        s1.end()

    # Span 2 — Rerank (mock)
    if trace:
        s2 = trace.span(name="rerank", input={"docs": [context]})
        s2.end(output={"top_doc": context})

    # Span 3 — Generate
    resp = client.chat.completions.create(
        model="local-model",
        messages=[{"role": "user",
                   "content": f"Context: {context}\nQ: {question}\nAnswer:"}],
        max_tokens=100,
    )
    answer = resp.choices[0].message.content.strip()

    if trace:
        s3 = trace.span(name="generate")
        s3.end(output={"answer": answer})
        trace.update(output={"answer": answer})
        lf.flush()

    elapsed = time.perf_counter() - t0
    print(f"✅ Traced call in {elapsed:.2f}s — {'visible in Langfuse UI' if LF_AVAILABLE else 'mock mode'}")
    return answer

ans = traced_rag_call("What is self-attention?")
print(ans[:200])


---
## 🟡 Exercise — Ex 06c-2: Prompt A/B Test with Langfuse

In [ ]:
show_exercise(
    "06c-2", "Prompt A/B test: 100 queries, 2 prompt versions",
    """Run 100 queries (50 per version):
  Version A — standard prompt: 'Answer this question: {question}'
  Version B — CoT prompt:      'Think step by step, then answer: {question}'
Log each call to Langfuse with tags ['prompt_vA'] or ['prompt_vB'].
Compare: avg response length, avg latency.
Conclusion: which version produced longer/better answers and at what cost?""",
    hints=[
        "Tag traces: trace.update(tags=['prompt_vA']) if LF_AVAILABLE",
        "Track result dict: {version, question, length, latency}",
        "Log to local list if Langfuse is not running",
    ],
    checks=[
        "100 queries total (50 per version)",
        "Langfuse traces created (or logged locally in mock mode)",
        "A/B comparison table printed",
    ],
)


In [ ]:
import statistics, time

PROMPT_A = "Answer this question concisely: {question}"
PROMPT_B = "Think step by step, then give a concise answer: {question}"

TEST_QUESTIONS = [
    "What is a transformer architecture?",
    "How does BERT differ from GPT?",
    "What is gradient descent?",
    "Explain the attention mechanism in one sentence.",
    "What is overfitting in machine learning?",
] * 10   # 50 questions × 2 versions = 100 total

ab_results = {"A": [], "B": []}

for version, template in [("A", PROMPT_A), ("B", PROMPT_B)]:
    for question in TEST_QUESTIONS:
        prompt = template.format(question=question)
        t0 = time.perf_counter()

        resp = client.chat.completions.create(
            model="local-model",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=150,
        )
        answer  = resp.choices[0].message.content.strip()
        latency = time.perf_counter() - t0

        if LF_AVAILABLE:
            trace = lf.trace(name="ab-test", input={"question": question, "version": version})
            trace.update(tags=[f"prompt_v{version}"], output={"answer": answer})
            lf.flush()

        ab_results[version].append({"answer": answer, "latency": latency, "length": len(answer)})

for v in ["A", "B"]:
    avg_len = statistics.mean(r["length"]  for r in ab_results[v])
    avg_lat = statistics.mean(r["latency"] for r in ab_results[v])
    print(f"Version {v}: avg_length={avg_len:.0f} chars | avg_latency={avg_lat:.2f}s")


In [ ]:
check([
    (len(ab_results["A"]) == 50, "50 queries for version A"),
    (len(ab_results["B"]) == 50, "50 queries for version B"),
    (all("answer"  in r for r in ab_results["A"] + ab_results["B"]), "All results have 'answer'"),
    (all("latency" in r for r in ab_results["A"] + ab_results["B"]), "All results have 'latency'"),
], exercise_id="06c-2")


In [ ]:
check([
    (len(ab_results["A"]) == 50, "Version A: 50 queries"),
    (len(ab_results["B"]) == 50, "Version B: 50 queries"),
], exercise_id="06c-final")
progress()
